In [ ]:
# mast3r parametres - как предсказанная сцена собирается в реконструкцию:

# Coarse LR - learning rate для выравнивания камер и 3D сцены. То есть насколько быстрыми
#  шагами оптимизация двигает камеры/точки, чтобы собрать сцену 

# Iterations for coarse alignment - кол-во итерация для первого (грубого) этапа оптимизации 
# база = 300, если сцена собирается плохо, можно попробовать 500

# Fine LR - learning rate для тонкой донастройки после грубого alignment

# Iterations for refinement
# база = 300, если сцена собирается плохо, можно попробовать 500

# OptLevel = refine+depth
# что именно MASt3R будет уточнять? варианты coarse/refine/refine+depth

# Matching Confidence Thr = 0
# насколько уверенными должны быть matches, чтобы использоваться? 
# 0 - почти все / 5-10 можно пробовать чтобы получить менее шумный / ломаный результат

# Shared intrinsics - у всех изображений одинаковые camera intrinsics: фокусное расстояние, параметры камеры и т.д.? 
# если да - все фото сняты одной камерой, без zoom, с одинаковым режимом

# Scenegraph 
# как MASt3R создаёт пары изображений для matching - сравнивать все или не все пары? 

# РЕЗУЛЬТАТ

# min_conf_thr - фильтрация по уверенности
# сlean-up depthmaps - попытка убрать шумные depth predictions

In [ ]:
# конвертация резуьтатов glb (может содержать и mesh и point cloud)

# ! pip install trimesh pygltflib

import trimesh
from pathlib import Path

input_path = Path("../outputs/exp_004_mast3r_bollard_001/exp_004_mast3r_bollard_001.glb.glb")
out_path = Path("../outputs/exp_004_mast3r_bollard_001/exp_004_mast3r_bollard_001.glb.ply")

obj = trimesh.load(input_path)

if isinstance(obj, trimesh.Scene):

    obj = trimesh.util.concatenate(list(obj.geometry.values()))

obj.export(out_path)

print(out_path.exists(), out_path)

True ../outputs/exp_004_mast3r_bollard_001/pointcloud.ply


In [ ]:
glb_path = Path("../outputs/exp_004_mast3r_bollard_001/exp_004_mast3r_bollard_001.glb.glb")

print("Exists:", glb_path.exists())
print("Size MB:", glb_path.stat().st_size / 1024 / 1024)

obj = trimesh.load(glb_path)

print("Loaded type:", type(obj))

Exists: True
Size MB: 22.399486541748047
Loaded type: <class 'trimesh.scene.scene.Scene'>


In [6]:
scene = trimesh.load(glb_path)

if isinstance(scene, trimesh.Scene):
    print("Scene geometries:", len(scene.geometry))
    
    for name, geom in scene.geometry.items():
        print("\n---")
        print("Name:", name)
        print("Type:", type(geom))
        
        if hasattr(geom, "vertices"):
            print("Vertices:", len(geom.vertices))
            print("Bounds:", geom.bounds)
        
        if hasattr(geom, "faces"):
            print("Faces:", len(geom.faces))
        
        if hasattr(geom, "colors"):
            print("Has colors:", geom.colors is not None)
else:
    print("Single geometry")
    print("Type:", type(scene))
    print("Vertices:", len(scene.vertices) if hasattr(scene, "vertices") else None)
    print("Faces:", len(scene.faces) if hasattr(scene, "faces") else None)
    print("Bounds:", scene.bounds if hasattr(scene, "bounds") else None)

Scene geometries: 25

---
Name: geometry_0
Type: <class 'trimesh.points.PointCloud'>
Vertices: 1214107
Bounds: [[-4.06738281 -2.8595922  -2.70616961]
 [ 2.14820695  2.25999284  5.62228489]]
Has colors: True

---
Name: geometry_1
Type: <class 'trimesh.base.Trimesh'>
Vertices: 4
Bounds: [[ 0.18376929 -0.34848732 -1.29318631]
 [ 0.43100122 -0.1023557  -1.2701546 ]]
Faces: 4

---
Name: geometry_2
Type: <class 'trimesh.base.Trimesh'>
Vertices: 14
Bounds: [[ 0.18376929 -0.34891987 -1.42972624]
 [ 0.43100122 -0.10192314 -1.26999283]]
Faces: 48

---
Name: geometry_3
Type: <class 'trimesh.base.Trimesh'>
Vertices: 4
Bounds: [[ 0.62281394 -0.52105552 -1.04349947]
 [ 0.80336815 -0.29105166 -0.95712095]]
Faces: 4

---
Name: geometry_4
Type: <class 'trimesh.base.Trimesh'>
Vertices: 14
Bounds: [[ 0.62190634 -0.52359933 -1.12398219]
 [ 0.80427575 -0.28850779 -0.95573449]]
Faces: 48

---
Name: geometry_5
Type: <class 'trimesh.base.Trimesh'>
Vertices: 4
Bounds: [[ 1.01725924 -0.47264802 -0.70657551]
 [ 

In [8]:
if isinstance(scene, trimesh.Scene):
    geometries = list(scene.geometry.values())
    # берем самую большую geometry по числу vertices
    main_geom = max(
        geometries,
        key=lambda g: len(g.vertices) if hasattr(g, "vertices") else 0
    )
else:
    main_geom = scene

print("Selected vertices:", len(main_geom.vertices))
print("Selected faces:", len(main_geom.faces) if hasattr(main_geom, "faces") else 0)

main_geom.export(out_path)
print("Saved:", out_path)

Selected vertices: 1214107
Selected faces: 0
Saved: ../outputs/exp_004_mast3r_bollard_001/pointcloud.ply


In [ ]:
# запуск из external/mast3r
# python3 demo.py \
#   --weights checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth \
#   --device cpu